In [ ]:
# imports 
import torch, os, json, clip
from tqdm import tqdm
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
model, preprocess = clip.load("ViT-B/32", device)

caps = json.load(open("captions_val2017.json"))
inst = json.load(open("instances_val2017.json"))

class_names = [c["name"] for c in sorted(inst["categories"], key = lambda x:x["id"])]
class_text = ["a photo of a " + c for c in class_names] 
class_tokens = clip.tokenize(class_text).to(device)

In [ ]:
count = 0

for fname in tqdm(sorted(os.listdir("val2017"))):

    image = Image.open("val2017/" + fname)
    image_in = preprocess(image).unsqueeze(0).to(device)

    img_id = int(fname.split(".")[0])
    caps5 = [a["captions"] for a in caps["annotations"] if a["image_id"]==img_id][:5]

    with torch.no_grad():
        img_feat = model.encode_image(image_in)
        cap_feat = model.encode_text(clip.tokenize(caps5).to(device))
        cls_feat = model.encode_text(class_tokens)
    
    img_feat = img_feat/img_feat.norm()
    cap_feat = cap_feat/cap_feat.norm(dim=-1, keepdim=True)
    cls_feat = cls_feat/cls_feat.norm(dim=-1, keepdim=True)

    best_caption = (img_feat @ cap_feat.T).argmax().item()
    top5 = (img_feat @ cls_feat.T).indices.squeeze().tolist()

    scores = (img_feat @ cls_feat.T).squeeze()
    threshold = 0.20
    multi_label = [class_names[i] for i,s in enumerate(scores) if s>=threshold]

    if count<=3:
        print("\n Image :", fname)
        print("Best Caption :", caps5[best_caption])
        print("Top 5 Classes:",[class_names[i] for i in top5])
        print("Multi Labels :", multi_label)
        count++
